In [8]:
import pandas as pd
import numpy as np

# File path
FILE_PATH = 'yellow_tripdata_2021-01.csv'

In [12]:
# Read only the first 100 rows into df
df = pd.read_csv(FILE_PATH, nrows=100)

print(df.head())

print(df.dtypes)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


## 2:Profile

In [15]:
chunk_size = 100000
total_rows = 0
null_counts = None
col_mins = {}
col_maxs = {}
col_dtypes = None

for df in pd.read_csv(FILE_PATH, chunksize=chunk_size, low_memory=False):
    # Convert date strings to datetime
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')
    
    if col_dtypes is None:
        col_dtypes = df.dtypes
        null_counts = df.isnull().sum()
    else:
        null_counts += df.isnull().sum()
        
    total_rows += len(df)
    
    for col in df.columns:
        if col == 'store_and_fwd_flag':
            s = df[col].dropna()
            if not s.empty:
                col_mins[col] = min(col_mins.get(col, s.min()), s.min())
                col_maxs[col] = max(col_maxs.get(col, s.max()), s.max())
        else:
            val_min, val_max = df[col].min(), df[col].max()
            if pd.notna(val_min):
                col_mins[col] = min(col_mins.get(col, val_min), val_min)
            if pd.notna(val_max):
                col_maxs[col] = max(col_maxs.get(col, val_max), val_max)

In [11]:
profile_table = pd.DataFrame({
    'dtype': col_dtypes,
    'null_count': null_counts,
    'null_pct': (null_counts / total_rows * 100).round(2),
    'min': pd.Series(col_mins),
    'max': pd.Series(col_maxs)
})

print(f"Total Rows Profiled: {total_rows}")
print(profile_table.to_string())

Total Rows Profiled: 1369765
                                dtype  null_count  null_pct                  min                  max
VendorID                        int64       98352      7.18                    1                    2
tpep_pickup_datetime   datetime64[ns]           0      0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime  datetime64[ns]           0      0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count                 int64       98352      7.18                    0                    8
trip_distance                 float64           0      0.00                  0.0            263163.28
RatecodeID                      int64       98352      7.18                    1                   99
store_and_fwd_flag             object       98352      7.18                    N                    Y
PULocationID                    int64           0      0.00                    1                  265
DOLocationID                    int64           0    

## 3:Find Problems

In [13]:
problem_counts = {
    'pickup_outside_jan_2021': 0,
    'dropoff_before_or_equal_pickup': 0,
    'distance_zero_or_negative': 0,
    'fare_zero_or_negative': 0,
    'passenger_count_zero': 0,
    'passenger_count_null': 0,
    'duplicate_rows': 0
}

dedup_cols = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount', 'total_amount'
]
seen_hashes = set()

for df in pd.read_csv(FILE_PATH, chunksize=chunk_size, low_memory=False):
    pickup = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')
    dropoff = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')
    
    # 1. Pickups outside Jan 2021
    outside_jan = (pickup < '2021-01-01') | (pickup >= '2021-02-01') | pickup.isna()
    problem_counts['pickup_outside_jan_2021'] += int(outside_jan.sum())
    
    # 2. Dropoff <= Pickup
    bad_duration = (dropoff <= pickup)
    problem_counts['dropoff_before_or_equal_pickup'] += int(bad_duration.sum())
    
    # 3. Trip distance <= 0
    bad_dist = (df['trip_distance'] <= 0) | df['trip_distance'].isna()
    problem_counts['distance_zero_or_negative'] += int(bad_dist.sum())
    
    # 4. Fare <= 0
    bad_fare = (df['fare_amount'] <= 0) | df['fare_amount'].isna()
    problem_counts['fare_zero_or_negative'] += int(bad_fare.sum())
    
    # 5. Passenger count == 0 or null
    problem_counts['passenger_count_zero'] += int((df['passenger_count'] == 0).sum())
    problem_counts['passenger_count_null'] += int(df['passenger_count'].isna().sum())
    
    # 6. Duplicates across chunks
    chunk_hashes = pd.util.hash_pandas_object(df[dedup_cols], index=False)
    for h in chunk_hashes:
        if h in seen_hashes:
            problem_counts['duplicate_rows'] += 1
        else:
            seen_hashes.add(h)

In [14]:
problems_table = pd.DataFrame(
    list(problem_counts.items()),
    columns=['Problem', 'Row_Count']
)

print(problems_table.to_string(index=False))

--- PROBLEM COUNTS ---
                       Problem  Row_Count
       pickup_outside_jan_2021         24
dropoff_before_or_equal_pickup       6557
     distance_zero_or_negative      19952
         fare_zero_or_negative       7411
          passenger_count_zero      26726
          passenger_count_null      98352
                duplicate_rows          0


## 4:Decide

In [17]:
decisions_data = [
    ["Duplicate rows", "Drop", "Duplicate logs inflate trip and revenue metrics.", "Data Engineering"],
    ["Pickup outside Jan 2021", "Drop", "Invalid dates (e.g., 2008/2009) corrupt monthly reporting partitions.", "Data Engineering"],
    ["Dropoff <= Pickup", "Drop", "Impossible trip duration indicating taximeter or clock errors.", "Data Engineering"],
    ["Fare amount <= 0", "Drop", "Represents cancellations, disputes, or meter errors, not completed revenue trips.", "Finance / Analytics"],
    ["Trip distance <= 0", "Drop", "Zero/negative distance indicates aborted trips or sensor failure.", "Operations"],
    ["Passenger count == 0 or NaN", "Flag & Fix", "Trip occurred with valid fare and distance; impute 1 and flag to preserve revenue data.", "Business / Analytics"]
]

decisions_table = pd.DataFrame(
    decisions_data,
    columns=["Problem", "Decision", "Why", "Rule Owner"]
)

print(decisions_table.to_string(index=False))

                    Problem   Decision                                                                                     Why           Rule Owner
             Duplicate rows       Drop                                        Duplicate logs inflate trip and revenue metrics.     Data Engineering
    Pickup outside Jan 2021       Drop                   Invalid dates (e.g., 2008/2009) corrupt monthly reporting partitions.     Data Engineering
          Dropoff <= Pickup       Drop                          Impossible trip duration indicating taximeter or clock errors.     Data Engineering
           Fare amount <= 0       Drop       Represents cancellations, disputes, or meter errors, not completed revenue trips.  Finance / Analytics
         Trip distance <= 0       Drop                       Zero/negative distance indicates aborted trips or sensor failure.           Operations
Passenger count == 0 or NaN Flag & Fix Trip occurred with valid fare and distance; impute 1 and flag to preserve

## 5:Clean

In [18]:
def clean_df(df: pd.DataFrame):
    # Operate on a copy to avoid mutating the original dataframe
    cleaned = df.copy()
    removed_counts = {}
    
    # Parse dates and nullable integers
    cleaned['tpep_pickup_datetime'] = pd.to_datetime(cleaned['tpep_pickup_datetime'], errors='coerce')
    cleaned['tpep_dropoff_datetime'] = pd.to_datetime(cleaned['tpep_dropoff_datetime'], errors='coerce')
    
    int_cols = ['VendorID', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'passenger_count']
    for col in int_cols:
        if col in cleaned.columns:
            cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce').astype('Int64')

    #Drop exact duplicates
    n_before = len(cleaned)
    cleaned = cleaned.drop_duplicates()
    removed_counts['duplicate_rows'] = n_before - len(cleaned)
    
    #Drop pickups outside Jan 2021
    n_before = len(cleaned)
    valid_jan = (cleaned['tpep_pickup_datetime'] >= '2021-01-01') & (cleaned['tpep_pickup_datetime'] < '2021-02-01')
    cleaned = cleaned[valid_jan]
    removed_counts['pickup_outside_jan_2021'] = n_before - len(cleaned)
    
    #Drop dropoff <= pickup
    n_before = len(cleaned)
    valid_duration = cleaned['tpep_dropoff_datetime'] > cleaned['tpep_pickup_datetime']
    cleaned = cleaned[valid_duration]
    removed_counts['dropoff_before_or_equal_pickup'] = n_before - len(cleaned)
    
    # Rule 4: Drop fare <= 0
    n_before = len(cleaned)
    cleaned = cleaned[cleaned['fare_amount'] > 0]
    removed_counts['fare_zero_or_negative'] = n_before - len(cleaned)
    
    #Drop distance <= 0
    n_before = len(cleaned)
    cleaned = cleaned[cleaned['trip_distance'] > 0]
    removed_counts['distance_zero_or_negative'] = n_before - len(cleaned)
    
    #Flag & Fix passenger count <= 0 or NaN (preserves records)
    cleaned['is_passenger_imputed'] = cleaned['passenger_count'].isna() | (cleaned['passenger_count'] <= 0)
    cleaned.loc[cleaned['is_passenger_imputed'], 'passenger_count'] = 1
    removed_counts['passenger_count_flagged_not_removed'] = 0
    
    return cleaned, removed_counts

In [19]:
total_before = 0
total_after = 0
total_removed_by_rule = {}

for df in pd.read_csv(FILE_PATH, chunksize=chunk_size, low_memory=False):
    total_before += len(df)
    cleaned_df, rule_dict = clean_df(df)
    total_after += len(cleaned_df)
    
    for rule, count in rule_dict.items():
        total_removed_by_rule[rule] = total_removed_by_rule.get(rule, 0) + count

In [23]:
summary_table = pd.DataFrame(
    list(total_removed_by_rule.items()),
    columns=['Rule', 'Rows_Removed']
)

print(f"Rows Before Cleaning: {total_before}")
print(f"Rows After Cleaning: {total_after}")
print(f"Total Rows Removed: {total_before - total_after}\n")
print(summary_table.to_string(index=False))

Rows Before Cleaning: 1369765
Rows After Cleaning: 1337533
Total Rows Removed: 32232

                               Rule  Rows_Removed
                     duplicate_rows             0
            pickup_outside_jan_2021            24
     dropoff_before_or_equal_pickup          6557
              fare_zero_or_negative          7343
          distance_zero_or_negative         18308
passenger_count_flagged_not_removed             0
